## Random Forest

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import numpy as np


In [ ]:
df = pd.read_csv("data/df_selected.csv")
df_clean = pd.read_csv("data/df_clean.csv")

In [ ]:
y = df_clean["taux_equipement_ve"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42
)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
print("Score train :", rf.score(X_train, y_train))
print("Score test :", rf.score(X_test, y_test))

Ce qui manque dans tes données :
prix du carburant local
subventions locales
urbanisation fine
disponibilité réelle des bornes
comportement des ménages

In [ ]:
importances = pd.Series(rf.feature_importances_, index=df.columns)
importances = importances.sort_values()

plt.figure(figsize=(8,6))
importances.plot(kind="barh")
plt.title("Importance des variables - Random Forest")
plt.show()

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE :", rmse)

L’erreur quadratique moyenne (RMSE) du modèle est d’environ 0,013, ce qui correspond à une erreur moyenne d’environ 1,3 point de pourcentage sur le taux d’équipement en véhicules électriques.
Étant donné que ce taux est en moyenne de l’ordre de 2 %, cette erreur reste significative, traduisant la difficulté à prédire précisément ce phénomène à l’échelle communale.
Toutefois, le modèle permet de capturer les grandes tendances et d’identifier les principaux facteurs explicatifs.

In [ ]:
df_result = pd.DataFrame({
    "réel": y_test,
    "prédit": y_pred
})

df_result.head()

In [ ]:
import shap

# explainer
explainer = shap.TreeExplainer(rf)

# calcul des shap values
shap_values = explainer.shap_values(X_test)

# graphique global
shap.summary_plot(shap_values, X_test)